In [ ]:
import glob, os, time, gc
import numpy as np, torch
from pathlib import Path
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

Image.MAX_IMAGE_PIXELS = None
SIZE   = 512
BATCH  = 2
WORK   = Path('/kaggle/working')
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEV == 'cuda', 'a 7B on CPU is not finishable -- attach a GPU'
print('device:', torch.cuda.get_device_name(0))
CKPT, DONE = WORK / 'd7b_ckpt.npy', WORK / 'd7b_done.npy'

DATA = Path('/kaggle/input/competitions/lost-in-the-museum-f1/archive/kaggle_dataset/kaggle_dataset')
if not DATA.exists():
    hits = [d for d in glob.glob('/kaggle/input/**/', recursive=True)
            if glob.glob(os.path.join(d, '*.png'))]
    assert hits, 'no PNG directory under /kaggle/input'
    DATA = Path(max(hits, key=lambda h: len(glob.glob(os.path.join(h, '*.png')))))
paths = sorted(DATA.glob('*.png'))
print(len(paths), 'images from', DATA)
assert len(paths) == 20000, f'expected 20000, got {len(paths)}'

In [ ]:
from transformers import AutoModel

HAVE_BNB = False

local = [d for d in glob.glob('/kaggle/input/**/', recursive=True)
         if os.path.exists(os.path.join(d, 'config.json'))]
print('model dirs:'); [print('   ', d) for d in local]
pick = sorted(d for d in local if '7b' in d.lower()) or\
       sorted(d for d in local if 'dinov3' in d.lower() and 'convnext' not in d.lower())
assert pick, 'attach the DINOv3 ViT-7B mirror'
MODEL = pick[0].rstrip('/'); print('using:', MODEL)

if HAVE_BNB:
    from transformers import BitsAndBytesConfig
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                           bnb_4bit_compute_dtype=torch.float16,
                           bnb_4bit_use_double_quant=True)
    model = AutoModel.from_pretrained(MODEL, device_map='auto',
                                      quantization_config=q).eval()
else:
    BATCH = 1
    model = AutoModel.from_pretrained(MODEL, device_map='auto',
                                      torch_dtype=torch.float16,
                                      low_cpu_mem_usage=True).eval()
assert getattr(model.config, 'model_type', '') != 'convnext', 'that is a ConvNeXt'
PATCH  = getattr(model.config, 'patch_size', 16)
HID    = getattr(model.config, 'hidden_size', 4096)
NPATCH = (SIZE // PATCH) ** 2
print(f'patch={PATCH} hidden={HID} tokens={NPATCH} -> block width {2*HID}')
print(f'GPU mem after load: {torch.cuda.memory_allocated()/2**30:.1f} GB')

In [ ]:
tf = transforms.Compose([
    transforms.Resize((SIZE, SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DS(Dataset):
    def __len__(self): return len(paths)
    def __getitem__(self, i):
        try: return tf(Image.open(paths[i]).convert('RGB')), i
        except Exception: return torch.zeros(3, SIZE, SIZE), i

@torch.no_grad()
def embed(x):
    h = model(pixel_values=x).last_hidden_state
    cls = h[:, 0].float()
    gem = h[:, -NPATCH:].float().clamp(min=1e-6).pow(3.0).mean(1).pow(1/3.0)
    return torch.cat([cls, gem], 1)

@torch.no_grad()
def embed_safe(x):
    try:
        return embed(x)
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        if len(x) == 1: raise
        h = len(x) // 2
        return torch.cat([embed_safe(x[:h]), embed_safe(x[h:])])

feats = np.load(CKPT) if CKPT.exists() else np.zeros((len(paths), 2*HID), np.float32)
done  = np.load(DONE) if DONE.exists() else np.zeros(len(paths), bool)
for pat in ('d7b_ckpt.npy', 'd7b_done.npy'):
    hit = sorted(glob.glob(f'/kaggle/input/**/{pat}', recursive=True))
    if hit and not CKPT.exists():
        a = np.load(hit[0])
        if pat.endswith('ckpt.npy') and a.shape == feats.shape: feats = a
        if pat.endswith('done.npy')  and a.shape == done.shape:  done = a
print(f'resuming with {done.sum()}/{len(paths)} embedded', flush=True)

todo = np.flatnonzero(~done)
dl = DataLoader(DS(), batch_size=BATCH, num_workers=2, sampler=todo.tolist())
t0 = time.time(); n = 0
for x, idx in dl:
    v = embed_safe(x.to(DEV).half()).cpu().numpy()
    if n == 0:
        assert np.isfinite(v).all(), 'first batch non-finite -- 4-bit overflow'
        print(f'first batch OK, block width {v.shape[1]}', flush=True)
    feats[idx.numpy()] = v; done[idx.numpy()] = True
    n += len(idx)
    if n % (BATCH * 50) == 0:
        r = n / (time.time() - t0)
        print(f'  {done.sum()}/{len(paths)}  {r:.1f} img/s  '
              f'ETA {(len(todo)-n)/r/60:.0f} min', flush=True)
        np.save(CKPT, feats); np.save(DONE, done)
        gc.collect(); torch.cuda.empty_cache()

np.save(CKPT, feats); np.save(DONE, done)
assert np.isfinite(feats).all(), 'non-finite features -- do not use'
np.save(WORK / f'features_d7b{SIZE}.npy', feats)
print(f'wrote features_d7b{SIZE}.npy {feats.shape} in {(time.time()-t0)/60:.1f} min')

In [ ]:
import pandas as pd

def l2(a, eps=1e-12):
    return a / (np.linalg.norm(a, axis=1, keepdims=True) + eps)

def whiten(x, dim, alpha=1.0):
    """numpy's gesdd driver does not converge on matrices this wide; fall back to
    gesvd and then to a covariance eigendecomposition, which is exact here
    because only V and the singular values are ever used."""
    mu = x.mean(0, keepdims=True); xc = x - mu
    try:
        _, s, vt = np.linalg.svd(xc, full_matrices=False)
    except np.linalg.LinAlgError:
        print('  gesdd did not converge -> gesvd', flush=True)
        try:
            from scipy.linalg import svd as sp_svd
            _, s, vt = sp_svd(xc, full_matrices=False, lapack_driver='gesvd')
        except Exception as e:
            print(f'  gesvd failed ({e}) -> covariance eigendecomposition', flush=True)
            w, V = np.linalg.eigh((xc.T @ xc) / (len(xc) - 1))
            o = np.argsort(-w); w, V = w[o], V[:, o]
            s = np.sqrt(np.maximum(w, 0)) * np.sqrt(len(xc) - 1); vt = V.T
    sc = (s[:dim] / np.sqrt(len(x) - 1)) ** alpha + 1e-8
    return l2(xc @ vt[:dim].T / sc).astype(np.float32)

assert np.isfinite(feats).all(), 'non-finite features'
f = whiten(l2(feats.astype(np.float64)), feats.shape[1], 1.0)
print('whitened', f.shape, flush=True)

names = np.array([p.name for p in paths])
df = pd.DataFrame(np.round(f, 5), columns=[f'feature_{i}' for i in range(f.shape[1])])
df.insert(0, 'image_name', names); df['ID'] = df['image_name']
assert len(df) == 20000 and not df.isna().any().any()
df.to_csv(WORK / 'submission.csv', index=False)
print(f'wrote submission.csv: {len(df)} rows x {df.shape[1]} cols  (expect LB 0.95302 = 284/298)')